# Data Exploration of different datasets

## Datasets used in this project

- [MARS dataset](https://www.sciencedirect.com/science/article/pii/S2352340923000604)
- [ITM-Rec dataset](https://arxiv.org/abs/2303.10230)

In [ ]:
import pandas as pd
from pandas.api.types import is_numeric_dtype
from sklearn.preprocessing import MinMaxScaler, OrdinalEncoder

In [ ]:
def preprocess(df: pd.DataFrame, target: str, user_col: str, item_col: str, ignored_cols: list[str]) -> pd.DataFrame:
    """Preprocess the main dataframe."""
    num_nan_targets = df[target].isna().sum()

    if num_nan_targets > 0:
        df = df.dropna(subset=[target])

    assert df[target].isna().sum() == 0

    df[user_col] = OrdinalEncoder().fit_transform(df[[user_col]])
    df[item_col] = OrdinalEncoder().fit_transform(df[[item_col]])

    protected_cols = set(
        ignored_cols + [user_col, item_col, target]
    )

    remaining_cols = [
        col for col in df.columns if col not in protected_cols
    ]

    # If the column is numeric, scale it
    # If the column is categorical, encode it (handle NaNs as "Undefined")
    for col in remaining_cols:
        if is_numeric_dtype(df[col]):
            df[col] = MinMaxScaler().fit_transform(df[[col]])
        else:
            df[col] = df[col].astype("category")
            num_nans = df[col].isna().sum()
            if num_nans > 0:
                df[col] = df[col].cat.add_categories("Undefined")
                df[col] = df[col].fillna("Undefined")

            df[col] = OrdinalEncoder().fit_transform(df[[col]])

    return df

## MARS Dataset

In [103]:
explicit_df_en = pd.read_csv("../data/mars_dataset/explicit_ratings_en.csv")
explicit_df_fr = pd.read_csv("../data/mars_dataset/explicit_ratings_fr.csv")

items_en = pd.read_csv("../data/mars_dataset/items_en.csv")
items_fr = pd.read_csv("../data/mars_dataset/items_fr.csv")

df_explicit = pd.concat([explicit_df_en, explicit_df_fr], ignore_index=True)
df_items = pd.concat([items_en, items_fr], ignore_index=True)

df_explicit["created_at"] = pd.to_datetime(df_explicit["created_at"])
df_items = df_items.drop(columns=["created_at"])

df = pd.merge(df_explicit, df_items, on="item_id", how="inner")

df.rename(
    columns={"Difficulty": "difficulty", "type": "item_type", "Software": "software"},
    inplace=True,
)

features = [
    "user_id",
    "item_id",
    "item_type",
    "difficulty",
    "nb_views",
    "watch_percentage",
    "description",
    "rating",
]

df = df[features]

df

,user_id,item_id,item_type,difficulty,nb_views,watch_percentage,description,rating
0,224557,510,tutorial,Beginner,1114.0,100,OneDrive for Businessis an online libraryto st...,10
1,224557,615,tutorial,Beginner,184.0,100,Tell me brings featuresand helps topic to your...,10
2,224557,7680,tutorial,Intermediate,73.0,100,The Groups calendar helps youto track all the ...,10
3,224293,510,tutorial,Beginner,1114.0,100,OneDrive for Businessis an online libraryto st...,10
4,224293,515,tutorial,Beginner,253.0,100,Once you sync your one drive library to your c...,10
...,...,...,...,...,...,...,...,...
88993,610452,419834,tutorial,NaN,42.0,92,NaN,10
88994,610452,419835,tutorial,NaN,37.0,100,NaN,10
88995,610452,419839,tutorial,NaN,33.0,99,NaN,10
88996,610452,419841,tutorial,NaN,37.0,100,NaN,10


In [105]:
num_nan_descriptions = df["description"].isna().sum()
print(f"Number of descriptions: {len(df['description'])}")
print(f"Number of descriptions with NaN: {num_nan_descriptions}")

Number of descriptions: 88998
Number of descriptions with NaN: 11203


In [100]:
df_preprocessed = preprocess(df, target="rating", user_col="user_id", item_col="item_id", ignored_cols=[])
df_preprocessed

,user_id,item_id,item_type,difficulty,nb_views,watch_percentage,rating
0,968.0,263.0,0.0,1.0,0.152759,1.00,10
1,968.0,295.0,0.0,1.0,0.025117,1.00,10
2,968.0,579.0,0.0,2.0,0.009882,1.00,10
3,943.0,263.0,0.0,1.0,0.152759,1.00,10
4,943.0,268.0,0.0,1.0,0.034587,1.00,10
...,...,...,...,...,...,...,...
88993,10594.0,2110.0,0.0,3.0,0.005627,0.92,10
88994,10594.0,2111.0,0.0,3.0,0.004941,1.00,10
88995,10594.0,2113.0,0.0,3.0,0.004392,0.99,10
88996,10594.0,2115.0,0.0,3.0,0.004941,1.00,10
